In [1]:
with open('input.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()

In [2]:
print("Length of dataset is : ", len(text))

Length of dataset is :  1115394


In [3]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [4]:
# Finding all unique characters in the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
# Create a mapping of chars and ints
stoi = {ch : i for i, ch in enumerate(chars)}
itos = {i : ch for i, ch in enumerate(chars)}
encode = lambda s : [stoi[c] for c in s] # Take a string and return int
decode = lambda l : ''.join([itos[i] for i in l]) # Take int and return string

print(decode(encode("Hi There")))

Hi There


In [6]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [7]:
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape)
data[:100]

torch.Size([1115394])


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [8]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1 : block_size + 1]

for t in range(block_size) :
    context = x[:t + 1]
    target = y[t]
    print(f"When context is {context} target is {target}")

When context is tensor([18]) target is 47
When context is tensor([18, 47]) target is 56
When context is tensor([18, 47, 56]) target is 57
When context is tensor([18, 47, 56, 57]) target is 58
When context is tensor([18, 47, 56, 57, 58]) target is 1
When context is tensor([18, 47, 56, 57, 58,  1]) target is 15
When context is tensor([18, 47, 56, 57, 58,  1, 15]) target is 47
When context is tensor([18, 47, 56, 57, 58,  1, 15, 47]) target is 58


In [11]:
batch_size = 4
block_size = 8

def get_batch(split) :
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print("Shape of xb is : ", xb.shape)
print(xb)
print("Shape of xb is : ", yb.shape)
print(yb)
print("--------------------------------------------")

for b in range(batch_size) : 
    for t in range(block_size) :
        context = xb[b][:t + 1]
        target = yb[b][t]
        print(f"When the context is {context.tolist()} the target is {target}")

Shape of xb is :  torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Shape of xb is :  torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
--------------------------------------------
When the context is [24] the target is 43
When the context is [24, 43] the target is 58
When the context is [24, 43, 58] the target is 5
When the context is [24, 43, 58, 5] the target is 57
When the context is [24, 43, 58, 5, 57] the target is 1
When the context is [24, 43, 58, 5, 57, 1] the target is 46
When the context is [24, 43, 58, 5, 57, 1, 46] the target is 43
When the context is [24, 43, 58, 5, 57, 1, 46, 43] the target is 39
When the context is [44] the target is 53
When the context is [44, 53] the target is 56
When the context 

In [13]:
class BiagramLanguageModel(nn.Module) :

    def __init__(self, vocab_size) :
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # Create a embedding table of size vocab_size * vocab_size

    def forward(self, idx, targets = None) :
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets == None :
            loss = None
        else :
            # Torch Cross entropy expects input is the form of (minibatch, C) : C should be the second dimension
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens) :

        for _ in range(max_new_tokens) :
            logits, loss = self(idx) # get logits : #B, T, C
            logits = logits[:, -1, :] # Take only T dimension : B * C
            probs = F.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples = 1) # Sample from the distribution, (B * 1)
            idx = torch.cat((idx, idx_next), dim = 1) # B, T + 1
        return idx

m = BiagramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(torch.zeros((1, 1), dtype = torch.long), max_new_tokens = 100)[0].tolist()))

torch.Size([32, 65])
tensor(5.0225, grad_fn=<NllLossBackward0>)

e:b&F:WoV?rky3kN&fvMpIm?hqR!vCD Gzsys:zORj !nsdUftTFO'Pov$po!r,EWmf;i$ sCXkCs :zbI-$qRbIFc,EaLAZ-pbo


In [15]:
# Create a optimiser :
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [16]:
batch_size = 32

for steps in range(5000) :
    xb, yb = get_batch('train')

    logtis, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.6209208965301514


In [18]:
print(decode(m.generate(torch.zeros((1, 1), dtype = torch.long), max_new_tokens = 300)[0].tolist()))


Chout mongl'gh cVe
YWSodocond.
Shem, ARWht ery high r aves an
NCE V:


sthack!N:
v
Hanck

O gr ckX: vFzMBift hicy,g t hqow'end t,e; m w yot; the doBE:
TELOUFaumy o t m t bonor Lag;
Fio sot y hyo'd w thio yondsoso rife heavepunf wngrMouicLUDUCHEWANTERe so y wd-d3adrk, ceDBAne, whyusF!R; w me majointe


Mathematical trci in attention : 

In [19]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [20]:
# Version 1 :
xbow = torch.zeros((B, T, C)) # X - bag of words
for b in range(B) :
    for t in range(T) :
        xprev = x[b, :t + 1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0)

In [21]:
# Version 2 :
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim = True)

xbow2 = wei @ x # (T, T) @ (B, T, C) ----> (B, T, T) @ (B, T, C) = (B, T, C)
torch.allclose(xbow, xbow2) # Giving false because of rounding off error, actual values very close

False

In [22]:
# Version 3 :
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = -1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3) # Giving false because of rounding off error, actual values very close

False

In [23]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias = False)
query = nn.Linear(C, head_size, bias = False)
value = nn.Linear(C, head_size, bias = False)
k = key(x) # (B, T, head_size)
q = query(x) # (B, T, head_size)
v = value(x)
wei = q @ k.transpose(-2, -1) # (B, T, head_size) @ (B, head_size, T) = (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = -1)
out = wei @ v

wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:

--->Attention is a communication mechanism. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.

--->There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.

--->Each example across batch dimension is of course processed completely independently and never "talk" to each other

--->In an "encoder" attention block just delete the single line that does masking with tril, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.

--->"self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)

--->"Scaled" attention additional divides wei by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [25]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a /= torch.sum(a, dim = 1, keepdim = True)
print(a)
print('----------------')
b = torch.randint(0, 10, (3, 2)).float()
print(b)
print('----------------')
c = a @ b
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
----------------
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
----------------
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [45]:
# Applying layer norm :
class BatchNorm1d :
    def __init__(self, dim, eps = 1e-5, momentum = 0.1) :
        # Declaration :
        self.eps = eps
        # Parameters :
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x) :
        # Forward pass :
        xmean = x.mean(1, keepdim = True)
        xvar = x.var(1, keepdim = True)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # Normalisation
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self) :
        return [self.gamma, self.beta]

torch.manual_seed(1337)
module = BatchNorm1d(100)
x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])

In [46]:
x[:, 0].std()

tensor(0.8803)

In [47]:
x[0, :].std()

tensor(1.0000)